# ConvNeXt V2 — Co-designing ConvNets with Masked Autoencoders

**Paper:** Woo, S., Debnath, S., Hu, R., Chen, X., Liu, Z., Kweon, I. S., & Xie, S. (2023). ConvNeXt V2: Co-designing and Scaling ConvNets with Masked Autoencoders. *CVPR 2023*. https://arxiv.org/abs/2301.00808

**ConvNeXt V1:** Liu, Z., et al. (2022). A ConvNet for the 2020s. *CVPR 2022*. https://arxiv.org/abs/2201.03545

**Implementation:** `hale_core/nn/vision/cnn/convnext.py`

> ConvNeXt V2 modernises the CNN design space by adding Global Response Normalization (GRN) — a channel competition mechanism that prevents feature collapse during masked autoencoding pre-training.

## Architecture Overview

```
Input [B, 3, H, W]
       │
  ┌────▼──────────────────────────────────────────┐
  │  STEM: 4×4 Conv stride-4 + LayerNorm2d        │  → [B, C0, H/4, W/4]
  └───────────────────────────────────────────────┘
       │
  ┌────▼──────────────────────────────────────────┐
  │  STAGE 0: N0 × ConvNeXtV2Block                │  → [B, C0, H/4, W/4]
  └───────────────────────────────────────────────┘
       │ Downsample (LN2d + 2×2 stride-2)
  ┌────▼──────────────────────────────────────────┐
  │  STAGE 1: N1 × ConvNeXtV2Block                │  → [B, C1, H/8, W/8]
  └───────────────────────────────────────────────┘
       │ Downsample
  ┌────▼──────────────────────────────────────────┐
  │  STAGE 2: N2 × ConvNeXtV2Block                │  → [B, C2, H/16, W/16]
  └───────────────────────────────────────────────┘
       │ Downsample
  ┌────▼──────────────────────────────────────────┐
  │  STAGE 3: N3 × ConvNeXtV2Block                │  → [B, C3, H/32, W/32]
  └───────────────────────────────────────────────┘
       │
  AdaptiveAvgPool → LayerNorm → Dropout → [B, C3]

ConvNeXtV2Block (for channels C, drop_path p):
  x → DWConv 7×7 (groups=C) → transpose [B,C,H,W]→[B,H,W,C]
    → LayerNorm → Linear(4C) → GELU
    → GRN(4C)  ← THE V2 ADDITION
    → Linear(C) → transpose [B,H,W,C]→[B,C,H,W]
    → StochasticDepth → (+residual x)

GRN (Global Response Normalization):
  x: [B, H, W, C]
  gx = ‖x‖₂ over (H,W) → [B, 1, 1, C]  (global spatial L2 norm per channel)
  nx = gx / mean(gx, dim=C) + ε          (normalise across channels)
  out = gamma * (x * nx) + beta + x       (scale, shift, add identity)
```

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(16, 11))
fig.patch.set_facecolor('#0d1117')
for ax in axes:
    ax.set_facecolor('#0d1117')
    ax.axis('off')

def box(ax, x, y, w, h, text, sub='', fc='#1f6feb', fontsize=9):
    r = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.08',
                        facecolor=fc, edgecolor='#58a6ff', lw=1.4)
    ax.add_patch(r)
    ax.text(x+w/2, y+h/2+(0.1 if sub else 0), text,
            ha='center', va='center', color='white',
            fontsize=fontsize, fontweight='bold', fontfamily='monospace')
    if sub:
        ax.text(x+w/2, y+h/2-0.18, sub, ha='center', va='center',
                color='#8b949e', fontsize=7)

def arr(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle='->', color='#58a6ff', lw=1.5))

# ── LEFT: ConvNeXtV2Block internal flow ──────────────────────────────────────
ax = axes[0]
ax.set_xlim(0, 9)
ax.set_ylim(0, 13)
ax.text(4.5, 12.5, 'ConvNeXtV2Block', ha='center', color='#f0f6fc',
        fontsize=12, fontweight='bold')

box(ax, 2, 11.5, 5, 0.65, 'Input x', '[B, C, H, W]', fc='#21262d')
box(ax, 2, 10.2, 5, 0.8,  'DWConv 7×7', 'groups=C  (pure spatial mixing)', fc='#1f6feb')
arr(ax, 4.5, 11.5, 4.5, 11.0)
box(ax, 2, 9.2,  5, 0.7,  'Transpose', '[B,C,H,W]→[B,H,W,C]', fc='#21262d')
arr(ax, 4.5, 10.2, 4.5, 9.9)
box(ax, 2, 8.2,  5, 0.7,  'LayerNorm', 'ε=1e-6, over C dim', fc='#388bfd')
arr(ax, 4.5, 9.2, 4.5, 8.9)
box(ax, 2, 7.2,  5, 0.7,  'Linear C → 4C', 'inverted bottleneck expand', fc='#1f6feb')
arr(ax, 4.5, 8.2, 4.5, 7.9)
box(ax, 2, 6.2,  5, 0.7,  'GELU', '', fc='#388bfd')
arr(ax, 4.5, 7.2, 4.5, 6.9)

# GRN — highlighted differently
box(ax, 1.5, 4.8, 6, 1.0, 'GRN (Global Response Norm)', 'V2 addition: channel feature competition\ngx=‖x‖₂→norm→gate', fc='#9e6a03', fontsize=8)
arr(ax, 4.5, 6.2, 4.5, 5.8)

box(ax, 2, 3.8,  5, 0.7,  'Linear 4C → C', 'contract', fc='#1f6feb')
arr(ax, 4.5, 4.8, 4.5, 4.5)
box(ax, 2, 2.8,  5, 0.7,  'Transpose', '[B,H,W,C]→[B,C,H,W]', fc='#21262d')
arr(ax, 4.5, 3.8, 4.5, 3.5)
box(ax, 2, 1.8,  5, 0.7,  'StochasticDepth', 'drop_path prob p', fc='#30363d')
arr(ax, 4.5, 2.8, 4.5, 2.5)
box(ax, 2, 0.8,  5, 0.7,  'Add (+residual x)', '[B, C, H, W]', fc='#2ea043')
arr(ax, 4.5, 1.8, 4.5, 1.5)
# skip arrow
ax.annotate('', xy=(2.5, 1.0), xytext=(1.2, 11.5),
            arrowprops=dict(arrowstyle='->', color='#f85149', lw=2,
                            connectionstyle='arc3,rad=0.0'))

# ── RIGHT: GRN detail ────────────────────────────────────────────────────────
ax = axes[1]
ax.set_xlim(0, 9)
ax.set_ylim(0, 13)
ax.text(4.5, 12.5, 'GRN — Global Response Normalization', ha='center',
        color='#f0f6fc', fontsize=11, fontweight='bold')

box(ax, 1.5, 11.2, 6, 0.8, 'Input x', '[B, H, W, 4C]  (after GELU)', fc='#21262d')

# Step 1: global L2 norm
box(ax, 1.5, 9.8, 6, 1.0, 'Step 1: Global Spatial L2 Norm',
    'gx = ‖x‖₂ over (H,W) → [B, 1, 1, 4C]\none scalar per channel per batch item', fc='#1f6feb', fontsize=8)
arr(ax, 4.5, 11.2, 4.5, 10.8)

# Step 2: normalise across channels
box(ax, 1.5, 8.4, 6, 1.0, 'Step 2: Channel Normalisation',
    'nx = gx / (mean(gx, dim=C) + ε)\nchannels with larger norm → larger weight', fc='#388bfd', fontsize=8)
arr(ax, 4.5, 9.8, 4.5, 9.4)

# Step 3: gate + residual
box(ax, 1.5, 7.0, 6, 1.0, 'Step 3: Scale + Residual',
    'out = γ·(x · nx) + β + x\nγ, β: learnable [4C], init 0', fc='#9e6a03', fontsize=8)
arr(ax, 4.5, 8.4, 4.5, 8.0)

box(ax, 1.5, 5.9, 6, 0.7, 'Output', '[B, H, W, 4C]', fc='#21262d')
arr(ax, 4.5, 7.0, 4.5, 6.6)

# Explanation
ax.text(4.5, 5.2, 'Why GRN helps FCMAE pre-training:', ha='center',
        color='#f0f6fc', fontsize=9, fontweight='bold')

explanations = [
    'Without GRN: channels collapse to similar features',
    '(feature redundancy → poor reconstruction signal)',
    '',
    'With GRN: channels compete via global norms',
    '→ diverse features, no redundancy',
    '→ analogous to Batch Norm between channel maps',
]
for i, e in enumerate(explanations):
    color = '#2ea043' if 'With GRN' in e else '#8b949e'
    ax.text(4.5, 4.7 - i*0.42, e, ha='center', color=color, fontsize=8)

# Model size table
ax.text(4.5, 2.3, 'Variant        depths        dims             embed_dim', ha='center',
        color='#58a6ff', fontsize=8, fontfamily='monospace')
rows = [
    'convnext_tiny   [3,3,9,3]    [96,192,384,768]    768',
    'convnext_base   [3,3,27,3]   [128,256,512,1024] 1024',
    'convnext_huge   [3,3,27,3]   [352,704,1408,2816] 2816',
]
for i, r in enumerate(rows):
    ax.text(4.5, 1.8 - i*0.45, r, ha='center', color='#8b949e', fontsize=7.5, fontfamily='monospace')

plt.suptitle('ConvNeXt V2 Architecture', fontsize=14, color='#f0f6fc',
             fontweight='bold', y=0.99)
plt.tight_layout()
plt.show()


## Key Innovations

### ConvNeXt V1 Modernisation Recipe (Liu et al. 2022)
Starting from ResNet-50, the V1 paper applied a series of ViT-inspired changes to a CNN:
- Stage ratio [1:1:3:1] → [1:1:3:1] (matching Swin)
- 3×3 conv → 7×7 depthwise conv (larger receptive field per layer)
- ReLU → GELU; BN → LayerNorm; more normalisation layers removed
- Inverted bottleneck (expand 4×) — opposite of Bottleneck
- Fewer activations: activation only after expand, not after each conv

Result: a plain CNN matching Swin Transformer accuracy at equivalent FLOPs.

### ConvNeXt V2 Additions (Woo et al. 2023)

**FCMAE Pre-training**: Apply masked autoencoding (like MAE for ViT) to the ConvNeXt backbone. Mask random 0.6 of patches at the pixel level, reconstruct from visible patches. However, applying FCMAE to V1 caused **feature collapse** — channels became redundant because there was nothing preventing them from learning the same features.

**Global Response Normalization (GRN)** is the fix:
- Compute per-channel global L2 norm (‖x‖₂ over H, W)
- Normalise these norms across channels
- Use normalised norms as gates → channels with larger global activation *enhance* themselves
- This is analogous to instance normalisation but operates on the *norms* (not the features directly)

**Why it works**: Channels that learn similar features will have similar global norms. GRN amplifies channels that stand out globally, effectively suppressing redundant channels. This encourages diverse feature learning, which is critical for reconstruction quality in FCMAE.

### Design Details
- **Channels-last convention**: after the 7×7 depthwise conv, the tensor is transposed to [B, H, W, C] for the LayerNorm and Linear layers, then transposed back. This avoids writing a custom BN wrapper and matches PyTorch's nn.LayerNorm interface.
- **LayerNorm2d**: a wrapper around `nn.LayerNorm` that handles the transpose (see lines 66–78 in `convnext.py`).
- **Stochastic depth**: rates distributed linearly from 0 at block 0 to `drop_path_rate` at the last block.
- **Stem**: 4×4 stride-4 conv (patch-like tokenisation), different from ResNet's 7×7+MaxPool. The stem is directly followed by LayerNorm2d.

## Component Walkthrough

### `GRN` (lines 83–115 in `convnext.py`)
```python
class GRN(nn.Module):
    def __init__(self, dim):
        self.gamma = nn.Parameter(torch.zeros(1, 1, 1, dim))  # init 0 → identity start
        self.beta  = nn.Parameter(torch.zeros(1, 1, 1, dim))

    def forward(self, x):  # x: [B, H, W, C]
        gx = x.norm(p=2, dim=(1, 2), keepdim=True)   # [B, 1, 1, C] global norms
        nx = gx / (gx.mean(dim=-1, keepdim=True) + 1e-6)  # normalise across C
        return self.gamma * (x * nx) + self.beta + x  # gate + residual
```

### `ConvNeXtV2Block` (lines 120–163 in `convnext.py`)
```python
def forward(self, x):           # x: [B, C, H, W]
    residual = x
    x = self.dwconv(x)          # 7×7 depthwise — spatial mixing
    x = x.permute(0, 2, 3, 1)  # channels-last for LN+Linear
    x = self.norm(x)            # LayerNorm over C
    x = self.pwconv1(x)         # Linear C→4C
    x = self.act(x)             # GELU
    x = self.grn(x)             # feature competition ← V2 key addition
    x = self.pwconv2(x)         # Linear 4C→C
    x = x.permute(0, 3, 1, 2)  # back to [B,C,H,W]
    return residual + self.drop_path(x)
```

### `ConvNeXtV2.forward` (lines 280–316 in `convnext.py`)
```python
def forward(self, x):
    x = self.stem(x)                # [B, dims[0], H/4, W/4]
    feature_maps = []
    for i, stage in enumerate(self.stages):
        x = stage(x)
        feature_maps.append(x)      # collect all 4 scale outputs
        if i < len(self.downsamples):
            x = self.downsamples[i](x)  # LN2d + 2×2 stride-2 conv
    pooled = self.norm(self.pool(feature_maps[-1]).flatten(1))
    return BackboneOutput(pooled_output=pooled, feature_maps=feature_maps)
```

In [ ]:
import sys
sys.path.insert(0, '../../src')

import torch
from hale_core.nn.vision import build_vision_backbone

x = torch.randn(1, 3, 224, 224)

for name in ['convnext_tiny', 'convnext_base', 'convnext_large']:
    bb = build_vision_backbone(name, img_size=224)
    bb.eval()
    with torch.no_grad():
        out = bb(x)
    print(f'{name:18s}  embed={out.pooled_output.shape[-1]}')
    print(f'  feature_maps: {[tuple(f.shape) for f in out.feature_maps]}')
    print(f'  params: {bb.num_parameters():,}')
    print()


## References

1. **[Primary]** Woo, S., Debnath, S., Hu, R., Chen, X., Liu, Z., Kweon, I. S., & Xie, S. (2023). **ConvNeXt V2: Co-designing and Scaling ConvNets with Masked Autoencoders**. *CVPR 2023*. https://arxiv.org/abs/2301.00808

2. Liu, Z., Mao, H., Wu, C.-Y., Feichtenhofer, C., Darrell, T., & Xie, S. (2022). **A ConvNet for the 2020s (ConvNeXt V1)**. *CVPR 2022*. https://arxiv.org/abs/2201.03545

3. He, K., Chen, X., Xie, S., Li, Y., Dollár, P., & Girshick, R. (2022). **Masked Autoencoders Are Scalable Vision Learners (MAE)**. *CVPR 2022*. — The MAE pre-training paradigm that FCMAE is based on.

4. Liu, Z., et al. (2021). **Swin Transformer**. *ICCV 2021*. — The ViT-based architecture that ConvNeXt competed against, providing the design blueprint.

5. Huang, G., Sun, Y., Liu, Z., Sedra, D., & Weinberger, K. Q. (2016). **Deep Networks with Stochastic Depth**. *ECCV 2016*. — Stochastic depth regularisation used in ConvNeXt V2.